In [2]:
import pennylane as qp
from pennylane.drawer.label import LabelledOp
from pennylane.fourier.mark import MarkedOp, mark

base = qp.PauliX(15)
op = LabelledOp(base, "one")
op2 = LabelledOp(base, "two")

assert op != op2

op = qp.X(0)
marked_op = mark(op, marker="my-x")
assert isinstance(marked_op, MarkedOp)
assert hasattr(marked_op, "marker")
assert marked_op.marker == "my-x"

print("works!")

works!


In [3]:
from pennylane.transforms import to_ppr
from pennylane import qnode, device, draw, state, qjit, specs
from pennylane.ops import PauliX, PauliZ, Hadamard

dev = device("lightning.qubit", wires=[0])


@qjit(capture=True)
@to_ppr
@qnode(dev)
def circuit(wire):
    Hadamard(wire)
    return state()


print(circuit(0))

print(specs(circuit)([0]))

[0.70710678+1.11022302e-16j 0.70710678+0.00000000e+00j]
Device: lightning.qubit
Device wires: 1
Shots: Shots(total=None)
Level: device

Wire allocations: 1
Total gates: 4
Gate counts:
- PauliRot-pi/2-w1: 3
- GlobalPhase: 1
Measurements:
- state(all wires): 1
Depth: 3


In [4]:
import numpy as np
import pennylane as qp


def equal_superposition(wires):
    for wire in wires:
        qp.Hadamard(wires=wire)


def oracle(wires, omega):
    qp.FlipSign(omega, wires=wires)


def num_grover_iterations(N, M):
    return np.ceil(np.sqrt(N / M) * np.pi / 4).astype(int)


def grover_circuit(num_qubits):
    wires = list(range(num_qubits))
    omega = np.array([np.zeros(num_qubits), np.ones(num_qubits)])

    M = len(omega)
    N = 2**num_qubits

    # Initial state preparation
    equal_superposition(wires)

    # Grover iterations
    for _ in range(num_grover_iterations(N, M)):
        for omg in omega:
            oracle(wires, omg)
        qp.templates.GroverOperator(wires)

    return qp.probs(wires=wires)

In [5]:
NUM_QUBITS = 12

dev = qp.device("default.qubit", wires=NUM_QUBITS)


@qp.qnode(dev)
def circuit_default_qubit():
    return grover_circuit(NUM_QUBITS)

In [6]:
circuit_default_qubit()

array([4.99100713e-01, 4.39319470e-07, 4.39319470e-07, ...,
       4.39319470e-07, 4.39319470e-07, 4.99100713e-01])

In [7]:
import pennylane as qp
from functools import partial

dev = qp.device("default.qubit")

bitstrings = ["01", "11", "11", "00", "01", "11", "11", "00"]

control_wires = [0, 1, 2]
target_wires = [3, 4]
work_wires = [5, 6]


@partial(qp.compile, basis_set="CNOT")
@qp.set_shots(1)
@qp.qnode(dev)
def circuit(index):
    qp.BasisState(index, wires=control_wires)
    #  added work wires below
    qp.QROM(bitstrings, control_wires, target_wires, work_wires, clean=False)
    return qp.sample(wires=control_wires + target_wires + work_wires)


print("Number of qubits: ", len(control_wires + target_wires + work_wires))
print("One-qubit gates: ", qp.specs(circuit)(0)["resources"].gate_sizes[1])
print("Two-qubit gates: ", qp.specs(circuit)(0)["resources"].gate_sizes[2])

Number of qubits:  7
One-qubit gates:  155
Two-qubit gates:  76


In [8]:
from pennylane.pauli import PauliSentence, PauliWord

ps = PauliSentence({PauliWord({0: "X", 1: "Y"}): 0.1, PauliWord({2: "Z", 0: "Y"}): -0.45j})

ps.prune(tol=0.2)
ps

(-0-0.45j) * Z(2) @ Y(0)

In [9]:
import pennylane as qp

w1 = qp.FermiWord({(0, 0): "+", (1, 1): "-"})
w2 = qp.FermiWord({(0, 1): "+", (1, 2): "-"})
s = qp.FermiSentence({w1: 0.1, w2: 3.1})
s.prune(0.05)
s

FermiSentence({FermiWord({(0, 0): '+', (1, 1): '-'}): 0.1, FermiWord({(0, 1): '+', (1, 2): '-'}): 3.1})

In [10]:
from jax import numpy as jnp
import pennylane as qp
import numpy as np

# For now, let's act on only one neutral atom
single_qubit_dev = qp.device("default.qubit", wires=1)

from pennylane.pulse import rydberg_drive as H_d

# Choose some arbitrary parameters
peak = 2
phase = np.pi / 2
detuning = 0

duration = 0.2  # We'll set all of our pulses' duration to 0.2


def blackman_window(peak, time):

    blackman = (
        peak * 0.42
        - 1 / 2 * peak * jnp.cos(2 * jnp.pi * time / duration)
        + peak * 0.08 * jnp.cos(4 * jnp.pi * time / duration)
    )

    return blackman


@qp.qnode(single_qubit_dev)
def state_evolution():

    # Use qp.evolve to find the final state of the atom after interacting with the pulse
    qp.evolve(H_d(blackman_window, phase, detuning, wires=[0]))([peak], t=[0, duration])

    return qp.state()


print("The final state is {}".format(state_evolution().round(2)))

The final state is [ 0.86+0.j -0.5 -0.j]


In [11]:
import pennylane as qp
from pennylane.ops import PhaseShift
from pennylane import numpy as pnp
from pennylane import qnode

dev = qp.device("default.qubit")


@qnode(dev, diff_method="parameter-shift", max_diff=2)
def circuit(x):
    qp.Hadamard(wires=0)
    PhaseShift(x, wires=0)
    return qp.expval(qp.PauliZ(0))


x = pnp.array(0.9, requires_grad=True)
circuit(x)

res = qp.grad(circuit)(x)

res

np.float64(-8.326672684688677e-17)

In [12]:
import pennylane as qp

dev = qp.device("default.qubit")

qp.decomposition.enable_graph()


@qp.decompose(gate_set=(qp.Toffoli, qp.PauliZ, qp.PauliX), max_expansion=1)
@qp.qnode(dev)
def circuit():
    qp.PauliZ(2)
    qp.TemporaryAND((0, 1, 2), (1, 1))
    return qp.state()


print(qp.specs(circuit)())

Device: default.qubit
Device wires: None
Shots: Shots(total=None)
Level: gradient

Wire allocations: 3
Total gates: 2
Gate counts:
- PauliZ: 1
- Toffoli: 1
Measurements:
- state(all wires): 1
Depth: 2


In [13]:
from pennylane.debugging import debug_probs
from pennylane.decomposition import register_resources, add_decomps
import numpy as np


class _DecomposingOp(qp.operation.Operation):
    """Define a minimal operator whose decomposition creates multiple gates.

    Specifically used to trigger the queue leaking bug when wrapped in qp.ctrl()."""

    num_wires = 1

    @staticmethod
    def compute_decomposition(wires):
        return [qp.RY(0.5, wires=wires), qp.RX(0.5, wires=wires)]


@register_resources({qp.RY: 1, qp.RX: 1})
def _my_decompose(wires, **__):
    qp.RY(0.5, wires=wires)
    qp.RX(0.5, wires=wires)


add_decomps(_DecomposingOp, _my_decompose)


@qp.qnode(qp.device("default.qubit", wires=[0, 1]))
def circuit():
    qp.H(0)
    qp.ctrl(_DecomposingOp, control=0)(wires=1)

    ops_before = list(qp.QueuingManager.active_context().queue)
    _ = debug_probs()
    ops_after = list(qp.QueuingManager.active_context().queue)
    assert ops_before == ops_after

    return qp.expval(qp.Z(0))


@qp.qnode(qp.device("default.qubit", wires=[0, 1]))
def correct_circuit():
    qp.H(0)
    qp.ctrl(_DecomposingOp, control=0)(wires=1)
    return qp.expval(qp.Z(0))


assert np.allclose(circuit(), correct_circuit())

In [14]:
from catalyst.device.decomposition import catalyst_decompose

gate_set = {
    "X",
    "CNOT",
    "Toffoli",
    "TemporaryAND",
    "Adjoint(TemporaryAND)",
    "Cond",
    "HybridAdjoint",
    "ForLoop",
    "S",
    "T",
    "Adjoint(S)",
    "Adjoint(T)",
    "RZ",
    "Hadamard",
    "GlobalPhase",
}


@qp.qjit(capture=False, static_argnums=2)
@catalyst_decompose(capabilities=None, target_gates=gate_set)
@qp.qnode(qp.device("lightning.qubit"))
def circuit(wires, work_wires, cvals):
    qp.MultiControlledX(
        wires,
        work_wires=work_wires,
        control_values=cvals,
        work_wire_type="borrowed",
    )
    return qp.probs(wires=wires)

In [15]:
mat = np.array(
    [
        [0.51378719 + 0.0j, 0.0546265 + 0.79145487j, -0.2051466 + 0.2540723j],
        [0.62651582 + 0.0j, -0.00828925 - 0.60570321j, -0.36704948 + 0.32528067j],
        [-0.58608928 + 0.0j, 0.03902657 + 0.04633548j, -0.57220635 + 0.57044649j],
    ]
)


@qp.qjit(capture=False)
@qp.qnode(qp.device("lightning.qubit", wires=range(3)))
def circuit_template(unitary_matrix, check=False):
    qp.BasisState(qp.math.array([1, 1, 0]), wires=[0, 1, 2])
    qp.BasisRotation(
        wires=range(3),
        unitary_matrix=unitary_matrix,
        check=check,
    )
    return qp.expval(qp.PauliZ(0) @ qp.PauliZ(1))


circuit_template(mat)

Array(0.30565861, dtype=float64)

In [16]:
import jax


def workflow(shots: int):
    @qp.transforms.cancel_inverses
    @qp.qnode(qp.device("default.qubit", wires=[0]), shots=shots)
    def aloha():
        qp.Hadamard(wires=0)
        return qp.sample()

    return aloha()


jaxpr = jax.make_jaxpr(workflow)(3)

j = jaxpr.jaxpr
assert j.outvars[0].aval.shape[0] is j.invars[0]
assert j.outvars[0].aval.shape[1] == 1

j2 = j.eqns[0].params["inner_jaxpr"]
assert j2.outvars[0].aval.shape[0] is j2.constvars[0]
assert j2.outvars[0].aval.shape[1] == 1

j3 = j2.eqns[0].params["inner_jaxpr"]
assert j3.outvars[0].aval.shape[0] is j3.constvars[0]
assert j3.outvars[0].aval.shape[1] == 1

TypeError: expected a sequence of integers or a single integer, got 'JitTracer<~int64[]>'

In [ ]:
qp.decomposition.enable_graph()


@qp.qjit(capture=False)
@qp.qnode(qp.device("null.qubit"))
def node(data, control_wires, target_wires, work_wires):
    qp.QROM(data, control_wires, target_wires, work_wires)
    return qp.state()


n = 4
m = 7
w = n
data = qp.math.array(np.random.choice(2, size=(2**n - 3, m)), like="jax")
control_wires = qp.math.arange(n, like="jax")
target_wires = qp.math.arange(m, like="jax")
work_wires = qp.math.arange(w, like="jax")
node(data, control_wires, target_wires, work_wires)

In [ ]:
from jax import numpy as jnp


@qp.qjit
@qp.qnode(qp.device("lightning.qubit", wires=4))
def f(sz):
    x = jnp.ones([sz], dtype=float)

    @qp.while_loop(lambda i, _: i < 3)
    def loop(i, a):
        return i + 1, a + x

    _, a2 = loop(1, x)
    return a2


result = f(3)
expected = 3 * jnp.ones(3)
assert qp.math.allclose(result, expected)

In [ ]:
import pennylane as qp

x_wires = [1, 2, 3]
output = [4, 5, 6]
work_wires_add = [7, 8]
control_wire = [0]
work_wires_ctrl = [9]
work_wire_type = "zeroed"


@qp.qjit
def func():
    return qp.ctrl(
        qp.SemiAdder(
            x_wires=x_wires,
            y_wires=output,
            work_wires=work_wires_add,
        ),
        control=control_wire,
        work_wires=work_wires_ctrl,
        work_wire_type=work_wire_type,
    )


op = func()
assert op.hyperparameters["work_wire_type"] == work_wire_type
assert op.work_wire_type == work_wire_type
assert op.control_wires == qp.wires.Wires(control_wire)
assert op.work_wires == qp.wires.Wires(work_wires_ctrl)

In [ ]:
import pennylane as qp

device = qp.device("default.qubit", wires=2)


@qp.set_shots(1000)
@qp.qnode(device)
def circuit():
    qp.Hadamard(wires=0)
    qp.CNOT(wires=[0, 1])
    qp.GlobalPhase(1 / 4)
    return qp.expval(qp.PauliZ(wires=0)), qp.expval(qp.PauliZ(wires=1))


loaded_circuit = qp.to_openqasm(circuit)()
print(loaded_circuit)

# get rid of any include lines in qasm as per issue
loaded_circuit = "\n".join(
    [line for line in loaded_circuit.split("\n") if not line.startswith("include")]
)
print(qp.from_qasm3(loaded_circuit)())

In [ ]:
import pennylane.estimator as qre

op = qre.RX()
qsvt = qre.QSVT(block_encoding=op, encoding_dims=(1, 1), poly_deg=5)

assert qsvt.num_wires == 1
assert qsvt.encoding_dims == (1, 1)
assert qsvt.block_encoding == qre.resource_rep(qre.RX)

In [ ]:
import pennylane as qp
import pennylane.numpy as np

n = 2
dev = qp.device("default.mixed", wires=2 * n)


@qp.qnode(dev)
def test(rho):
    # Only initialize n of the 2n qubits using with rho
    qp.QubitDensityMatrix(rho, wires=range(0, n))

    # Apply Hadamard gate to ancilla qubits
    for a in range(n, 2 * n):
        qp.H(a)

    return qp.probs(wires=range(n))


GHZ = [1 / np.sqrt(2), 0, 0, 1 / np.sqrt(2)]
rho = qp.math.dm_from_state_vector(GHZ)
test(rho)

In [ ]:
from pennylane.devices.qubit_mixed.apply_operation import _phase_shift
import numpy as np

rho = np.array([[0.5, 0.5], [0.5, 0.5]])
# bra side (axis 0)
rho = _phase_shift(rho, axis=0)
# ket side (axis 1, conjugate phase — same for Z since conj(-1) = -1)
# rho = _phase_shift(rho, axis=1)
rho

In [ ]:
from pennylane.operation import Operation
import pennylane as qp
import numpy as np


class CustomOp(Operation):  # pylint: disable=too-few-public-methods
    resource_keys = set()

    @property
    def resource_params(self) -> dict:
        return {}

    def compute_matrix(self):
        return np.array([[1, 0], [0, 1]])


@qp.decompose(strict=True)
@qp.qnode(qp.device("default.qubit"))
def circuit():
    CustomOp(0)
    return qp.state()


circuit()

In [ ]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose(gate_set=(qp.H, qp.Y))
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.Toffoli([0, 1, 2])
    return qp.state()


circuit()

In [ ]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.ops.op_math.exp(qp.X(0), 1)
    return qp.state()


qp.specs(circuit)()

In [1]:
from pennylane.operation import Operation
import pennylane as qp
import numpy as np

qp.decomposition.enable_graph()


class CustomOp(Operation):  # pylint: disable=too-few-public-methods
    resource_keys = set()

    @property
    def resource_params(self) -> dict:
        return {}

    def compute_matrix(self):
        return np.array([[1, 0], [0, 1]])


@qp.qnode(qp.device("null.qubit"))
def circuit():
    CustomOp(0)
    return qp.state()


circuit()

array([1., 0.])

In [1]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose(gate_set=(qp.PauliRot,))
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.ops.op_math.Evolution(qp.X(0), 0.5)
    return qp.state()


circuit()

array([0.87758256+0.j        , 0.        -0.47942554j])

In [1]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose(gate_set=(qp.PhaseShift, qp.GlobalPhase))
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.RZ(0.5, 0)
    return qp.state()


circuit()

array([0.96891242-0.24740396j, 0.        +0.j        ])

In [3]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose(gate_set=(qp.H,), stopping_condition=lambda op: isinstance(op, qp.RZ))
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.RZ(0.5, 0)
    return qp.state()


circuit()

array([0.96891242-0.24740396j, 0.        +0.j        ])

In [2]:
import pennylane as qp

qp.decomposition.enable_graph()


@qp.decompose
@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.ControlledQubitUnitary(qp.Toffoli([0, 1, 2]).matrix(), wires=[0, 1, 2])
    return qp.state()


circuit()

array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])

In [1]:
from pennylane.operation import Operator
import pennylane as qp


class CustomOp(Operator):  # pylint: disable=too-few-public-methods
    pass


assert not qp.decomposition.has_decomp(CustomOp)


@qp.register_resources({qp.RZ: 2, qp.CNOT: 1})
def custom_decomp(theta, wires, **__):
    qp.RZ(theta, wires=wires[0])
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.RZ(theta, wires=wires[0])


@qp.register_resources({qp.RX: 2, qp.CZ: 1})
def custom_decomp2(theta, wires, **__):
    qp.RX(theta, wires=wires[0])
    qp.CZ(wires=[wires[0], wires[1]])
    qp.RX(theta, wires=wires[0])


@qp.register_resources({qp.RY: 2, qp.CNOT: 1})
def custom_decomp3(theta, wires, **__):
    qp.RY(theta, wires=wires[0])
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.RY(theta, wires=wires[0])


with qp.decomposition.local_decomps():

    qp.add_decomps(CustomOp, custom_decomp)
    qp.add_decomps(CustomOp, custom_decomp2, custom_decomp3)
    qp.add_decomps(qp.CRX, custom_decomp)

    assert qp.decomposition.has_decomp(CustomOp)
    assert qp.decomposition.has_decomp(CustomOp(wires=[0, 1]))
    assert len(qp.list_decomps(CustomOp)) == 3
    assert custom_decomp in qp.list_decomps(qp.CRX)

# test that the context properly cleans up.
assert len(qp.list_decomps(CustomOp)) == 0
assert not qp.decomposition.has_decomp(CustomOp)
assert custom_decomp not in qp.list_decomps(qp.CRX)

In [1]:
from scipy.stats import unitary_group
import pennylane as qp
import numpy as np

dev = qp.device("default.qutrit", wires=3)


def ansatz(phi, theta, omega, U):
    qp.TShift(0)
    qp.TAdd([0, 1])
    qp.TRX(phi, wires=2, subspace=(0, 1))
    qp.TClock(1)
    qp.TRY(theta, wires=0, subspace=(1, 2))
    qp.TSWAP([0, 2])
    qp.QutritUnitary(U, wires=[2, 1])
    qp.ControlledQutritUnitary(U, wires=[2, 1], control_wires=[0])
    qp.TRZ(omega, wires=1, subspace=(0, 2))


@qp.qnode(dev)
def circuit():
    phi, theta, omega = np.random.rand(3) * 2 * np.pi
    U = unitary_group.rvs(9, random_state=10)

    ansatz(phi, theta, omega, U)
    qp.adjoint(ansatz)(phi, theta, omega, U)
    return qp.state()


expected = np.zeros(27)
expected[0] = 1
res = circuit()

assert np.allclose(res, expected)

In [1]:
from pennylane.devices.preprocess import decompose
import pennylane as qp


@qp.register_resources({qp.RX: 1})
def dummy_rz_decomp_1(theta, wires):
    qp.RX(theta, wires)


num_work_wires = 1
fixed_decomps = {qp.RZ: [dummy_rz_decomp_1]}
alt_decomps = None
exp_types = [qp.RX]

target_gates = {"Hadamard": 1, "RX": 5, "RY": 5, "PhaseShift": 5, "GlobalPhase": 0}

tape = qp.tape.QuantumScript([qp.RZ(0.2, 0)])

batch, _ = decompose(
    tape,
    lambda obj: obj.name in target_gates,
    device_wires=None,
    num_work_wires=num_work_wires,
    target_gates=target_gates,
    fixed_decomps=fixed_decomps,
    alt_decomps=alt_decomps,
)
new_tape = batch[0]

for op, exp_t in zip(new_tape.operations, exp_types, strict=True):
    assert isinstance(op, exp_t)
assert all(op.name in target_gates for op in new_tape.operations)

DeviceError: Operator RZ(0.2, wires=[0]) not supported with device and does not provide a decomposition.

In [3]:
import pennylane as qp

qasm_string = """
qubit q0;
qubit q1;
qubit q2;

float theta = 0.2;
int power = 2;

ry(theta / 2) q0;
rx(theta) q1;
pow(power) @ x q0;

def random(qubit q) -> bit
{
    bit b = "0";
    h q;
    measure q -> b;
    return b;
}

bit m = random(q2);

if (m) {
    int i = 0;
    while (i < 5) {
        i = i + 1;
        rz(i) q1;
    }
}
"""


@qp.qnode(qp.device("default.qubit", wires=[0, 1, 2]))
def my_circuit():
    qp.from_qasm3(qasm_string, {"q0": 0, "q1": 2, "q2": 1})()
    return qp.expval(qp.Z(0))


print(qp.draw(my_circuit)())

0: ──RY(0.10)──X²────────────────────────────────────────────────────┤  <Z>
1: ──H─────────┤↗├───────────────────────────────────────────────────┤     
2: ──RX(0.20)───║───RZ(1.00)──RZ(2.00)──RZ(3.00)──RZ(4.00)──RZ(5.00)─┤     
                ╚═══╩═════════╩═════════╩═════════╩═════════╝              


In [22]:
import numpy as np

dev = qp.device("default.qubit", wires=4)


@qp.qnode(dev, diff_method="parameter-shift")
def circuit(x, y):
    qp.RX(x[0], wires=0)
    qp.Toffoli(wires=(0, 1, 2))
    qp.CRY(x[1], wires=(0, 1))
    qp.Rot(x[2], x[3], y, wires=0)
    return qp.expval(qp.Z(0)), qp.expval(qp.X(1))


x = qp.numpy.array([0.05, 0.1, 0.2, 0.3], requires_grad=True)
y = qp.numpy.array(0.4, requires_grad=False)
specs_func = qp.specs(circuit)
specs_func(x, y)

Device: default.qubit
Device wires: 4
Shots: Shots(total=None)
Level: gradient

Wire allocations: 3
Total gates: 4
Gate counts:
- RX: 1
- Toffoli: 1
- CRY: 1
- Rot: 1
Measurements:
- expval(PauliZ): 1
- expval(PauliX): 1
Depth: 4


In [1]:
import pennylane as qp
import pennylane.estimator as qre

qp.decomposition.enable_graph()

gate_set = {"T", "CNOT", "S", "Hadamard", "RZ", "GlobalPhase", "Toffoli", "MidMeasure"}

num_control_wires = 3
data = ["010", "111", "110", "000", "010", "111", "110", "000"]
bitstring_size = 3

wires = qp.registers({"intermediate_result": num_control_wires, "final_result": bitstring_size})

dev = qp.device("default.qubit")


@qp.set_shots(1)
@qp.decompose(gate_set=gate_set)
@qp.qnode(dev)
def circuit():

    qp.SelectOnlyQRAM(
        data,
        control_wires=wires["intermediate_result"],
        target_wires=wires["final_result"],
    )

    return qp.probs(wires=wires["final_result"])


print(circuit())
print(qp.estimator.estimate(circuit, gate_set=gate_set)())

[0. 0. 1. 0. 0. 0. 0. 0.]
--- Resources: ---
 Total wires: 7
   algorithmic wires: 6
   allocated wires: 1
     zero state: 1
     any state: 0
 Total gates : 168
   'Toffoli': 24,
   'CNOT': 12,
   'S': 48,
   'Hadamard': 84
